
# Exploração, Qualidade e Tratamento de Dados (Camada Raw -> Silver)

**Objetivo:** Ingestão da camada Raw, auditoria de qualidade dos dados, tratamento de nulos e tipos, validação de regras de negócio e persistência na camada Silver.


## Inspeção Inicial e Verificações de Qualidade
**Procedimentos realizados:**
- **Mapeamento Estrutural:** Análise de tipos originais e volume de registros (50.000 linhas) via `df.info()`.
- **Auditoria de Unicidade:** Checagem de integridade da chave primária (`order_id`) para confirmar se 1 linha = 1 pedido.

In [0]:

import pandas as pd


# Carregar os dados da camada Raw
df = pd.read_csv("/Volumes/workspace/default/dados_entregas/raw/E-commerce_Delivery_Shipping_Data_2026.csv")

# NSPECAO INICIAL
display(df.info())
display(df.head())
display(df.dtypes)

#VERIFICAR DUPLICADAS
duplicatas_linhas = df.duplicated().sum()
print("Linhas totalmente duplicadas:", duplicatas_linhas)

#VERIFICAR DUPLICADAS POR ID
duplicatas_pedidos = df["order_id"].duplicated().sum()
print("IDs de pedidos duplicados:", duplicatas_pedidos)



## Tratamento e Limpeza de Dados
**Problemas encontrados e tratamentos aplicados:**
- **Tipagem de Data:** A coluna `order_date` estava armazenada como texto (`object`). Ela foi convertida para `datetime` padronizada no nível de dia, eliminando a exibição de horários e preparando a base para análises temporais.

- **Valores Ausentes em `return_reason`:** Identificados nulos na coluna `return_reason`. A ausência de dado decorre do fluxo operacional: pedidos sem solicitação de devolução (`return_requested` = No) não possuem motivo. Aplicada imputação contextual com `'N/A'`.



**Validação de Regra de Negócio — Indicador de Atraso (`late_delivery`)**

**Objetivo:** Garantir a consistência lógica entre o indicador categórico de atraso e o cálculo de dias.
- **Regra de Negócio:** Um pedido é considerado em atraso quando a entrega real supera a prometida (`delivery_delay_days > 0`), devendo ter a flag `late_delivery` = `'Yes'`.

- **Confirmação de Ausência de Atrasos Negativos:** O teste (df['delivery_delay_days'] < 0).any() retornou False, confirmando a inexistência de prazos incoerentes na base.


- **Validação Cruzada entre Flag e Cálculo:** A filtragem condicional verificando divergências entre o prazo calculado (delivery_delay_days > 0) e o indicador registrado (late_delivery != 'Yes') retornou 0 linhas, confirmando 100% de consistência na lógica de atrasos em toda a base.

In [0]:


# CONVERTE PARA DATETIME GARANTINDO QUE O TEMPO FIQUE ZERADO NO NIVEL DE DIA
df['order_date'] = pd.to_datetime(df['order_date']).dt.floor('d')

display(df.head())
df.info()


#VERIFICAR RELACAO ENTRE COLUNAS RETURN_REQUESTED E RETURN_REASON PARA TRATAR NULOS
display(
    df.groupby("return_requested")["return_reason"].count()
)

#NULOS IDENTIFICADOS POR AUSENCIA DE SOLICITACAO DE DEVOLUCAO
#TRATAR NULOS
df["return_reason"] = df["return_reason"].fillna("N/A")
df.info()


# VERIFICAR SE O INDICADOR DE ATRASO DA COLUNA late_delivery ESTA CORRETO
display(
    df[[
        "delivery_delay_days",
        "late_delivery"
    ]].head(15)
)

# VERIFICAR TODA A COLUNA  
df.loc[
    ((df["delivery_delay_days"] > 0) & (df["late_delivery"] != "Yes")) |
    ((df["delivery_delay_days"] <= 0) & (df["late_delivery"] != "No")),
    ["delivery_delay_days", "late_delivery"]
]

# VERIFICAR DADOS NEGATIVOS NAS COLUNAS DE PRAZOS
print("Valores negativos:",((df['delivery_delay_days'] < 0).any()))


display(
    df[[
        "promised_delivery_days",
        "actual_delivery_days",
        "delivery_delay_days"
    ]].head(20)
)

# VERIFICAR CONSISTENCIA DO CALCULO DIAS DE ATRASO, CHECAR ATRASO REGISTRADO COM ATRASO CALCULADO
atraso_calculado = (
            df["actual_delivery_days"] - df["promised_delivery_days"]
        ).clip(lower=0)
(df["delivery_delay_days"] != atraso_calculado).sum()





##  Exportação para a Camada Silver
**Persistência:** Escrita da base limpa e tratada no Volume Silver do Unity Catalog para criação da tabela consumida pelas análises SQL.

In [0]:
# EXPORTAR DF PARA CSV
df.to_csv(
    "/Volumes/workspace/default/dados_entregas/silver/entregas_tratadas.csv",
    index=False
)